Importacion de librerias

In [5]:
import pandas as pd
import joblib
import optuna

from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

Leyendo el csv limpio, el pipeline y ver la distribucion

In [6]:
df = pd.read_csv("../data/interim/employee_performance_clean.csv")

X = df.drop(columns=["attrition"])
y = df["attrition"]

print("Distribución de clases:")
print(y.value_counts())
print(y.value_counts(normalize=True).round(3))

preproc = joblib.load("../models/preprocessing_pipeline.pkl")

Distribución de clases:
attrition
0    11840
1     2160
Name: count, dtype: int64
attrition
0    0.846
1    0.154
Name: proportion, dtype: float64


Configuración del cross validation. Se empleará junto a la optuna

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (11200, 19)
X_test: (2800, 19)


In [8]:

def evaluar_test(nombre, modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)

    return {
        "modelo": nombre,
        "accuracy_test": accuracy_score(y_test, y_pred),
        "precision_test": precision_score(y_test, y_pred, zero_division=0),
        "recall_test": recall_score(y_test, y_pred, zero_division=0),
        "f1_test": f1_score(y_test, y_pred, zero_division=0)
    }

Optuna para KNN

In [9]:
def objective_knn(trial):
    params = {
        "n_neighbors": trial.suggest_int("n_neighbors", 3, 40),
        "weights": trial.suggest_categorical("weights", ["uniform", "distance"]),
        "metric": trial.suggest_categorical("metric", ["euclidean", "manhattan"])
    }

    pipe = Pipeline([
        ("preproc", preproc),
        ("smote", SMOTE(random_state=42)),
        ("clf", KNeighborsClassifier(**params))
    ])

    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring="recall",
        n_jobs=-1
    )

    return scores.mean()


study_knn = optuna.create_study(direction="maximize")
study_knn.optimize(objective_knn, n_trials=30)

print("KNN - Mejor recall CV:", study_knn.best_value)
print("KNN - Mejores params:", study_knn.best_params)

[I 2026-05-03 01:27:12,102] A new study created in memory with name: no-name-b7386b88-6f42-45d9-9db8-9966454b9ea2
[I 2026-05-03 01:27:17,256] Trial 0 finished with value: 0.8009416101197955 and parameters: {'n_neighbors': 39, 'weights': 'distance', 'metric': 'euclidean'}. Best is trial 0 with value: 0.8009416101197955.
[I 2026-05-03 01:27:20,325] Trial 1 finished with value: 0.6354243109659043 and parameters: {'n_neighbors': 12, 'weights': 'distance', 'metric': 'euclidean'}. Best is trial 0 with value: 0.8009416101197955.
[I 2026-05-03 01:27:21,391] Trial 2 finished with value: 0.3651570746418698 and parameters: {'n_neighbors': 25, 'weights': 'distance', 'metric': 'manhattan'}. Best is trial 0 with value: 0.8009416101197955.
[I 2026-05-03 01:27:22,627] Trial 3 finished with value: 0.353584652760325 and parameters: {'n_neighbors': 34, 'weights': 'uniform', 'metric': 'manhattan'}. Best is trial 0 with value: 0.8009416101197955.
[I 2026-05-03 01:27:23,702] Trial 4 finished with value: 0.3

KNN - Mejor recall CV: 0.80151796933903
KNN - Mejores params: {'n_neighbors': 39, 'weights': 'uniform', 'metric': 'euclidean'}


In [10]:
best_knn = Pipeline([
    ("preproc", preproc),
    ("smote", SMOTE(random_state=42)),
    ("clf", KNeighborsClassifier(**study_knn.best_params))
])

best_knn.fit(X_train, y_train)
joblib.dump(best_knn, "../models/tuned_knn_optuna.pkl")

C:\Users\Usuario\miniforge3\envs\ProjectG7\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


['../models/tuned_knn_optuna.pkl']

Optuna para Regresion Lineal

In [11]:
def objective_lr(trial):
    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    C = trial.suggest_float("C", 0.001, 100, log=True)

    pipe = Pipeline([
        ("preproc", preproc),
        ("smote", SMOTE(random_state=42)),
        ("clf", LogisticRegression(
            penalty=penalty,
            C=C,
            solver="liblinear",
            random_state=42,
            max_iter=1000
        ))
    ])

    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring="recall",
        n_jobs=-1
    )

    return scores.mean()


study_lr = optuna.create_study(direction="maximize")
study_lr.optimize(objective_lr, n_trials=30)

print("LR - Mejor recall CV:", study_lr.best_value)
print("LR - Mejores params:", study_lr.best_params)

[I 2026-05-03 01:30:20,321] A new study created in memory with name: no-name-75b2f63b-ec80-41a3-b90d-2482a113f84a
[I 2026-05-03 01:30:20,536] Trial 0 finished with value: 0.4739800619921253 and parameters: {'penalty': 'l2', 'C': 5.785857787283408}. Best is trial 0 with value: 0.4739800619921253.
[I 2026-05-03 01:30:20,748] Trial 1 finished with value: 0.4606735360643378 and parameters: {'penalty': 'l2', 'C': 0.008476892820655712}. Best is trial 0 with value: 0.4739800619921253.
[I 2026-05-03 01:30:20,886] Trial 2 finished with value: 0.4739800619921253 and parameters: {'penalty': 'l1', 'C': 79.09533658272804}. Best is trial 0 with value: 0.4739800619921253.
[I 2026-05-03 01:30:21,077] Trial 3 finished with value: 0.47282231716511686 and parameters: {'penalty': 'l1', 'C': 0.47242844748524904}. Best is trial 0 with value: 0.4739800619921253.
[I 2026-05-03 01:30:21,269] Trial 4 finished with value: 0.46068191337857084 and parameters: {'penalty': 'l2', 'C': 0.0029295907106112475}. Best is 

LR - Mejor recall CV: 0.5034966909608779
LR - Mejores params: {'penalty': 'l1', 'C': 0.0069269741731821655}


In [12]:
best_lr = Pipeline([
    ("preproc", preproc),
    ("smote", SMOTE(random_state=42)),
    ("clf", LogisticRegression(
        penalty=study_lr.best_params["penalty"],
        C=study_lr.best_params["C"],
        solver="liblinear",
        random_state=42,
        max_iter=1000
    ))
])

best_lr.fit(X_train, y_train)
joblib.dump(best_lr, "../models/tuned_lr_optuna.pkl")

C:\Users\Usuario\miniforge3\envs\ProjectG7\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Usuario\miniforge3\envs\ProjectG7\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


['../models/tuned_lr_optuna.pkl']

Optuna para SVM

In [13]:
def objective_svm(trial):
    kernel = trial.suggest_categorical("kernel", ["linear", "rbf"])
    C = trial.suggest_float("C", 0.01, 100, log=True)

    params = {
        "kernel": kernel,
        "C": C
    }

    if kernel == "rbf":
        params["gamma"] = trial.suggest_categorical("gamma", ["scale", "auto"])

    pipe = Pipeline([
        ("preproc", preproc),
        ("smote", SMOTE(random_state=42)),
        ("clf", SVC(**params))
    ])

    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring="recall",
        n_jobs=-1
    )

    return scores.mean()


study_svm = optuna.create_study(direction="maximize")
study_svm.optimize(objective_svm, n_trials=20)

print("SVM - Mejor recall CV:", study_svm.best_value)
print("SVM - Mejores params:", study_svm.best_params)

[I 2026-05-03 01:30:34,294] A new study created in memory with name: no-name-03107ad0-2a19-4b04-920a-c98d0098de38
[I 2026-05-03 01:30:50,161] Trial 0 finished with value: 0.5347608276786462 and parameters: {'kernel': 'linear', 'C': 0.013474955412015164}. Best is trial 0 with value: 0.5347608276786462.
[I 2026-05-03 01:52:18,875] Trial 1 finished with value: 0.5272346485716679 and parameters: {'kernel': 'linear', 'C': 88.92107288939043}. Best is trial 0 with value: 0.5347608276786462.
[I 2026-05-03 01:55:20,155] Trial 2 finished with value: 0.5272346485716679 and parameters: {'kernel': 'linear', 'C': 11.617259491720898}. Best is trial 0 with value: 0.5347608276786462.
[I 2026-05-03 01:55:37,623] Trial 3 finished with value: 0.5312741894948478 and parameters: {'kernel': 'linear', 'C': 0.0870377046203457}. Best is trial 0 with value: 0.5347608276786462.
[I 2026-05-03 01:55:54,458] Trial 4 finished with value: 0.5364999581134289 and parameters: {'kernel': 'linear', 'C': 0.1405172072712012}

SVM - Mejor recall CV: 0.5376627293289771
SVM - Mejores params: {'kernel': 'linear', 'C': 0.2540912900517698}


In [14]:
svm_params = {
    "kernel": study_svm.best_params["kernel"],
    "C": study_svm.best_params["C"]
}

if study_svm.best_params["kernel"] == "rbf":
    svm_params["gamma"] = study_svm.best_params["gamma"]

best_svm = Pipeline([
    ("preproc", preproc),
    ("smote", SMOTE(random_state=42)),
    ("clf", SVC(**svm_params))
])

best_svm.fit(X_train, y_train)
joblib.dump(best_svm, "../models/tuned_svm_optuna.pkl")

['../models/tuned_svm_optuna.pkl']

Optuna para Random Forest

In [15]:
def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_categorical("max_depth", [5, 10, 20, None]),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None])
    }

    pipe = Pipeline([
        ("preproc", preproc),
        ("smote", SMOTE(random_state=42)),
        ("clf", RandomForestClassifier(
            **params,
            random_state=42,
            n_jobs=-1
        ))
    ])

    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring="recall",
        n_jobs=-1
    )

    return scores.mean()


study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(objective_rf, n_trials=30)

print("RF - Mejor recall CV:", study_rf.best_value)
print("RF - Mejores params:", study_rf.best_params)

[I 2026-05-03 02:06:16,762] A new study created in memory with name: no-name-db98a382-0cbb-4378-8429-67c6b340e252
[I 2026-05-03 02:06:27,624] Trial 0 finished with value: 0.0 and parameters: {'n_estimators': 498, 'max_depth': 20, 'min_samples_split': 13, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.0.
[I 2026-05-03 02:06:33,242] Trial 1 finished with value: 0.0 and parameters: {'n_estimators': 295, 'max_depth': 20, 'min_samples_split': 12, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.0.
[I 2026-05-03 02:06:42,169] Trial 2 finished with value: 0.0 and parameters: {'n_estimators': 413, 'max_depth': None, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.0.
[I 2026-05-03 02:06:44,965] Trial 3 finished with value: 0.027784200385356457 and parameters: {'n_estimators': 272, 'max_depth': 5, 'min_samples_split': 9, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 3

RF - Mejor recall CV: 0.03878026304766692
RF - Mejores params: {'n_estimators': 144, 'max_depth': 5, 'min_samples_split': 12, 'min_samples_leaf': 6, 'max_features': 'sqrt'}


In [16]:
best_rf = Pipeline([
    ("preproc", preproc),
    ("smote", SMOTE(random_state=42)),
    ("clf", RandomForestClassifier(
        **study_rf.best_params,
        random_state=42,
        n_jobs=-1
    ))
])

best_rf.fit(X_train, y_train)
joblib.dump(best_rf, "../models/tuned_rf_optuna.pkl")

['../models/tuned_rf_optuna.pkl']

Comparacion de resultados Optuna

In [17]:
optuna_results = pd.DataFrame([
    {
        "modelo": "knn",
        "best_recall_cv_optuna": study_knn.best_value,
        "best_params_optuna": study_knn.best_params,
        "path": "../models/tuned_knn_optuna.pkl"
    },
    {
        "modelo": "lr",
        "best_recall_cv_optuna": study_lr.best_value,
        "best_params_optuna": study_lr.best_params,
        "path": "../models/tuned_lr_optuna.pkl"
    },
    {
        "modelo": "svm",
        "best_recall_cv_optuna": study_svm.best_value,
        "best_params_optuna": study_svm.best_params,
        "path": "../models/tuned_svm_optuna.pkl"
    },
    {
        "modelo": "rf",
        "best_recall_cv_optuna": study_rf.best_value,
        "best_params_optuna": study_rf.best_params,
        "path": "../models/tuned_rf_optuna.pkl"
    }
])

optuna_results = optuna_results.sort_values(
    "best_recall_cv_optuna",
    ascending=False
)

display(optuna_results)

,modelo,best_recall_cv_optuna,best_params_optuna,path
0,knn,0.801518,"{'n_neighbors': 39, 'weights': 'uniform', 'met...",../models/tuned_knn_optuna.pkl
2,svm,0.537663,"{'kernel': 'linear', 'C': 0.2540912900517698}",../models/tuned_svm_optuna.pkl
1,lr,0.503497,"{'penalty': 'l1', 'C': 0.0069269741731821655}",../models/tuned_lr_optuna.pkl
3,rf,0.038780,"{'n_estimators': 144, 'max_depth': 5, 'min_sam...",../models/tuned_rf_optuna.pkl


Comparacion antes y despues del tuning

In [18]:
baseline_results = pd.DataFrame({
    "modelo": ["knn", "lr", "svm", "rf"],
    "recall_baseline_sprint3": [0.5856, 0.4768, 0.2106, 0.0000]
})

In [19]:
import numpy as np

comparison = baseline_results.merge(
    optuna_results[["modelo", "best_recall_cv_optuna"]],
    on="modelo",
    how="left"
)

comparison["mejora_abs"] = (
    comparison["best_recall_cv_optuna"] 
    - comparison["recall_baseline_sprint3"]
)

denominador = comparison["recall_baseline_sprint3"].replace(0, np.nan)

comparison["mejora_pct"] = (
    comparison["mejora_abs"] / denominador * 100
).round(2)

comparison = comparison.sort_values("best_recall_cv_optuna", ascending=False)

display(comparison)

,modelo,recall_baseline_sprint3,best_recall_cv_optuna,mejora_abs,mejora_pct
0,knn,0.5856,0.801518,0.215918,36.87
2,svm,0.2106,0.537663,0.327063,155.30
1,lr,0.4768,0.503497,0.026697,5.60
3,rf,0.0000,0.038780,0.038780,NaN


Evaluar mejor modelo optuna en test

In [20]:
best_model_name = optuna_results.iloc[0]["modelo"]

best_models = {
    "knn": best_knn,
    "lr": best_lr,
    "svm": best_svm,
    "rf": best_rf
}

best_model = best_models[best_model_name]

print("Mejor modelo Optuna:", best_model_name)

Mejor modelo Optuna: knn


In [21]:
test_metrics = evaluar_test(
    best_model_name,
    best_model,
    X_test,
    y_test
)

display(pd.DataFrame([test_metrics]))

y_pred_best = best_model.predict(X_test)

print(classification_report(y_test, y_pred_best, zero_division=0))

,modelo,accuracy_test,precision_test,recall_test,f1_test
0,knn,0.282857,0.149154,0.775463,0.250187


              precision    recall  f1-score   support

           0       0.82      0.19      0.31      2368
           1       0.15      0.78      0.25       432

    accuracy                           0.28      2800
   macro avg       0.49      0.48      0.28      2800
weighted avg       0.72      0.28      0.30      2800



Guardando resultados

In [22]:
Path("../models").mkdir(parents=True, exist_ok=True)

optuna_results.to_csv("../models/optuna_results.csv", index=False)
comparison.to_csv("../models/tuning_comparison_baseline_vs_optuna.csv", index=False)

print("Resultados guardados correctamente.")

Resultados guardados correctamente.


In [24]:
from imblearn.over_sampling import SMOTE
import pandas as pd

#Aplicando preprocesamiento al TRAIN
X_train_pre = preproc.fit_transform(X_train)

#Aplicando smote
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train_pre, y_train)

#Mostrando distribución
print("Distribución ANTES de SMOTE:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True).round(3))

print("\nDistribución DESPUÉS de SMOTE:")
print(pd.Series(y_res).value_counts())
print(pd.Series(y_res).value_counts(normalize=True).round(3))

Distribución ANTES de SMOTE:
attrition
0    9472
1    1728
Name: count, dtype: int64
attrition
0    0.846
1    0.154
Name: proportion, dtype: float64

Distribución DESPUÉS de SMOTE:
attrition
0    9472
1    9472
Name: count, dtype: int64
attrition
0    0.5
1    0.5
Name: proportion, dtype: float64
